In [1]:
# First, you will need to install CoolProp if you haven't already:
# !pip install CoolProp

import CoolProp.CoolProp as CP

def isentropic_expansion(fluid, P1_bar, T1_C, P2_bar):
    """
    Calculates the isentropic exit state for a turbine/compressor.
    """
    print(f"--- ISENTROPIC PROCESS SOLVER ---")
    print(f"Fluid: {fluid}")
    
    # Convert inputs to SI units (Pa and K) for CoolProp
    P1_Pa = P1_bar * 1e5
    T1_K = T1_C + 273.15
    P2_Pa = P2_bar * 1e5

    # 1. Calculate State 1 (Inlet) Properties
    h1 = CP.PropsSI('H', 'P', P1_Pa, 'T', T1_K, fluid) / 1000 # Convert J/kg to kJ/kg
    s1 = CP.PropsSI('S', 'P', P1_Pa, 'T', T1_K, fluid) / 1000 # Convert J/kg-K to kJ/kg-K
    
    print(f"\nState 1 (Inlet):")
    print(f"  P1 = {P1_bar} bar")
    print(f"  T1 = {T1_C}°C")
    print(f"  h1 = {h1:.2f} kJ/kg")
    print(f"  s1 = {s1:.4f} kJ/kg-K")

    # 2. Define State 2 (Exit) via Isentropic Constraint (s2 = s1)
    s2 = s1
    s2_J = s2 * 1000 # Convert back to standard SI for CoolProp

    # 3. Calculate State 2 Properties
    T2_K = CP.PropsSI('T', 'P', P2_Pa, 'S', s2_J, fluid)
    T2_C = T2_K - 273.15
    h2 = CP.PropsSI('H', 'P', P2_Pa, 'S', s2_J, fluid) / 1000
    
    # Check phase/quality (CoolProp returns -1 for subcooled, values > 1 for superheated)
    x2 = CP.PropsSI('Q', 'P', P2_Pa, 'S', s2_J, fluid)
    
    print(f"\nState 2 (Ideal Isentropic Exit):")
    print(f"  P2 = {P2_bar} bar")
    print(f"  s2 = {s2:.4f} kJ/kg-K  <-- (s2 = s1)")
    print(f"  T2 = {T2_C:.2f}°C")
    print(f"  h2 = {h2:.2f} kJ/kg")
    
    if 0 <= x2 <= 1:
        print(f"  Phase: Two-Phase Liquid-Vapor Mixture")
        print(f"  Quality (x) = {x2:.4f}")
    elif x2 < 0:
        print(f"  Phase: Subcooled Liquid")
    else:
        print(f"  Phase: Superheated Vapor")
        
    # Calculate ideal work (W_rev = \Delta H)
    w_rev = h2 - h1
    print(f"\nIdeal Turbine Work (W_rev): {w_rev:.2f} kJ/kg")

    return h1, h2, w_rev

# --- Run the HP Turbine Example from Lecture 19 ---
# Inlet: 60 bar, 500 C. Outlet: 10 bar.
h_in, h_out, work = isentropic_expansion('Water', P1_bar=60, T1_C=500, P2_bar=10)

--- ISENTROPIC PROCESS SOLVER ---
Fluid: Water

State 1 (Inlet):
  P1 = 60 bar
  T1 = 500°C
  h1 = 3423.11 kJ/kg
  s1 = 6.8826 kJ/kg-K

State 2 (Ideal Isentropic Exit):
  P2 = 10 bar
  s2 = 6.8826 kJ/kg-K  <-- (s2 = s1)
  T2 = 239.78°C
  h2 = 2920.40 kJ/kg
  Phase: Subcooled Liquid

Ideal Turbine Work (W_rev): -502.71 kJ/kg
